In [1]:
import json
from pathlib import Path
from datetime import datetime

def save_analysis_json(analysis_obj, out_file):
    """
    Save analysis object (dict) to JSON file.
    """
    out_file = Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    out_file.write_text(json.dumps(analysis_obj, indent=2, ensure_ascii=False), encoding="utf-8")
    return str(out_file)

def _safe_get_visual_type(visual_dict):
    """
    Visuals can be like:
      {"VisualId": "...", "KPIVisual": {...}}
    or sometimes already like:
      {"KPIVisual": {...}}
    """
    if not isinstance(visual_dict, dict):
        return "Unknown", {}
    # find first key that endswith Visual
    for k, v in visual_dict.items():
        if k.endswith("Visual") and isinstance(v, dict):
            return k, v

    return "Unknown", {}

def generate_html_dashboard_from_analysis_obj(analysis_obj, out_file="dashboard_preview.html"):
    """
    Generate a local HTML preview from a QuickSight Analysis dict.
    No AWS required.
    """
    definition = analysis_obj.get("Definition", {})
    sheets = definition.get("Sheets", [])

    html = []
    html.append("<!doctype html><html><head><meta charset='utf-8'>")
    html.append("<meta name='viewport' content='width=device-width, initial-scale=1'>")
    html.append("<title>Dashboard Preview</title>")
    html.append("""
    <style>
      body { font-family: Arial, sans-serif; margin:0; background:#f6f7fb; color:#111; }
      .topbar { background:white; border-bottom:1px solid #e5e7eb; padding:12px 16px; position:sticky; top:0; z-index:10; }
      .title { font-size:18px; font-weight:700; }
      .sub { color:#6b7280; font-size:12px; margin-top:3px; }
      .tabs { display:flex; gap:8px; margin-top:10px; flex-wrap:wrap; }
      .tab { padding:8px 10px; background:#f3f4f6; border:1px solid #e5e7eb; border-radius:10px; cursor:pointer; font-size:13px; }
      .tab.active { background:#111827; color:white; border-color:#111827; }
      .container { padding:16px; max-width:1200px; margin:0 auto; }
      .sheet { display:none; }
      .sheet.active { display:block; }
      .grid { display:grid; grid-template-columns: repeat(12, 1fr); gap:12px; }
      .card { background:white; border:1px solid #e5e7eb; border-radius:14px; padding:12px; box-shadow:0 2px 8px rgba(0,0,0,.04); }
      .card h3 { margin:0 0 8px 0; font-size:14px; }
      .meta { color:#6b7280; font-size:12px; line-height:1.4; }
      .badge { display:inline-block; padding:3px 8px; border:1px solid #e5e7eb; border-radius:999px; font-size:11px; color:#374151; background:#f9fafb; margin-right:6px; }
      .warn { background:#fff7ed; border:1px solid #fed7aa; color:#9a3412; padding:10px; border-radius:12px; margin-top:10px; }
      .small { font-size:12px; color:#6b7280; margin-top:10px; }
      /* simple layout sizing */
      .span-3 { grid-column: span 3; }
      .span-4 { grid-column: span 4; }
      .span-6 { grid-column: span 6; }
      .span-12 { grid-column: span 12; }
    </style>
    """)
    html.append("</head><body>")

    # header
    html.append("<div class='topbar'>")
    html.append("<div class='title'>Local Dashboard Preview</div>")
    html.append("<div class='sub'>Generated from QuickSight Definition (local preview)</div>")

    # tabs
    html.append("<div class='tabs'>")
    for i, s in enumerate(sheets):
        name = s.get("Name", f"Sheet {i+1}")
        html.append(f"<div class='tab' data-sheet='{i}'>{name}</div>")
    html.append("</div>")

    # warning box if TextBoxVisual present
    has_textbox = False
    for s in sheets:
        for v in s.get("Visuals", []):
            if isinstance(v, dict) and ("TextBoxVisual" in v):
                has_textbox = True
                break
    if has_textbox:
        html.append("<div class='warn'><b>Warning</b>: TextBoxVisual detected (titles/annotations). This is OK in preview, but some API deployments may exclude them.</div>")

    html.append("</div>")  # end topbar
    html.append("<div class='container'>")

    # sheets
    for i, s in enumerate(sheets):
        name = s.get("Name", f"Sheet {i+1}")
        html.append(f"<div class='sheet' id='sheet-{i}'>")
        html.append(f"<h2 style='margin:6px 0 12px 0;'>{name}</h2>")
        html.append("<div class='grid'>")

        visuals = s.get("Visuals", [])
        if not visuals:
            html.append("<div class='card span-12'><h3>No visuals</h3><div class='meta'>This sheet has no visuals.</div></div>")
        else:
            for v in visuals:
                if not isinstance(v, dict):
                    continue

                # detect type
                vtype, inner = _safe_get_visual_type(v)
                wrapper_id = v.get("VisualId", "")
                inner_id = inner.get("VisualId", "")

                # title extraction
                title = ""
                if vtype == "TextBoxVisual":
                    title = inner.get("ChartConfiguration", {}).get("TextBoxChartConfiguration", {}).get("Content", "")
                else:
                    t = inner.get("Title", {})
                    if isinstance(t, dict) and t.get("Visibility") == "VISIBLE":
                        title = t.get("FormatText", {}).get("PlainText", "")

                if not title:
                    title = vtype

                # sizing 
                span_class = "span-6"
                if vtype in ("TableVisual",):
                    span_class = "span-12"
                elif vtype in ("KPIVisual",):
                    span_class = "span-3"

                html.append(f"<div class='card {span_class}'>")
                html.append(f"<h3>{title}</h3>")
                html.append("<div class='meta'>")
                html.append(f"<span class='badge'>{vtype}</span>")
                if wrapper_id:
                    html.append(f"<span class='badge'>wrapper_id: {wrapper_id}</span>")
                if inner_id and inner_id != wrapper_id:
                    html.append(f"<span class='badge'>inner_id: {inner_id}</span>")
                html.append("</div>")
                html.append("</div>")

        html.append("</div>") 
        html.append("</div>") # sheet

    html.append("<div class='small'>Tip: Use the two configs to prove generalization: different roles ⇒ different previews.</div>")
    html.append("</div>")  

    html.append("""
    <script>
      const tabs = document.querySelectorAll('.tab');
      const sheets = document.querySelectorAll('.sheet');
      function activate(i){
        tabs.forEach(t => t.classList.remove('active'));
        sheets.forEach(s => s.classList.remove('active'));
        const tab = document.querySelector(`.tab[data-sheet="${i}"]`);
        const sheet = document.getElementById(`sheet-${i}`);
        if(tab) tab.classList.add('active');
        if(sheet) sheet.classList.add('active');
      }
      tabs.forEach(t => t.addEventListener('click', () => activate(t.dataset.sheet)));
      activate(0);
    </script>
    """)
    html.append("</body></html>")

    out_file = Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    out_file.write_text("\n".join(html), encoding="utf-8")
    return str(out_file)
